# Read FIG.csv file and find isomeric SMILES
## FRAGRANCE INGREDIENT GLOSSARY
## https://d3t14p1xronwr0.cloudfront.net/docs/ifra-fragrance-ingredient-glossary-april-2020.pdf

In [1]:
import time
import urllib.parse
from typing import Optional, Tuple, Any, Dict

import pandas as pd
import requests

IN_CSV  = "FIG.csv"
OUT_CSV = "FIG_with_isomeric_smiles.csv"
SLEEP_BETWEEN_CALLS = 0.12   
TIMEOUT = 15

PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
PUGVIEW_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "SMILESFetcher/2.0 (dataset prep)", "Accept": "application/json"})

# ───── HTTP helpers ─────
def _get_json(url: str, retries: int = 2, backoff: float = 0.6) -> Tuple[Optional[dict], int]:
    last = -1
    for i in range(retries + 1):
        try:
            r = SESSION.get(url, timeout=TIMEOUT)
            last = r.status_code
            if r.status_code == 200:
                return r.json(), r.status_code
            if r.status_code in (400, 404):
                return None, r.status_code
        except requests.RequestException:
            last = -1
        time.sleep(backoff * (2 ** i))
    return None, last

# ───── Extractors ─────
def _extract_smiles_from_property(j: Dict[str, Any]) -> Optional[str]:
    try:
        props = j["PropertyTable"]["Properties"][0]
        return props.get("IsomericSMILES") or props.get("CanonicalSMILES")
    except Exception:
        return None

def _extract_smiles_from_pugview(j: Dict[str, Any]) -> Optional[str]:
    # Walk PUG-View JSON to find a SMILES string
    targets = {"ISOMERIC SMILES", "CANONICAL SMILES", "SMILES"}
    found = {}

    def pull_value(info: dict) -> Optional[str]:
        sv = info.get("StringValue")
        if isinstance(sv, str) and sv:
            return sv.strip()
        val = info.get("Value")
        if isinstance(val, dict):
            swm = val.get("StringWithMarkup")
            if isinstance(swm, list) and swm:
                s = swm[0].get("String")
                if isinstance(s, str) and s:
                    return s.strip()
        s = info.get("String")
        return s.strip() if isinstance(s, str) and s else None

    def walk(node):
        if isinstance(node, dict):
            label = None
            nm = node.get("Name")
            if isinstance(nm, str):
                label = nm.strip().upper()
            th = node.get("TOCHeading")
            if isinstance(th, str):
                label = th.strip().upper() or label
            if label in targets:
                info_list = node.get("Information")
                if isinstance(info_list, list):
                    for info in info_list:
                        if isinstance(info, dict):
                            val = pull_value(info)
                            if val:
                                found.setdefault(label, val)
            for v in node.values():
                walk(v)
        elif isinstance(node, list):
            for it in node:
                walk(it)

    walk(j)
    for key in ("ISOMERIC SMILES", "CANONICAL SMILES", "SMILES"):
        if key in found:
            return found[key]
    return None

# ───── PubChem lookups ─────
def pc_props_from_RN(cas: str) -> Optional[str]:
    cas = (cas or "").strip()
    if not cas:
        return None
    url = f"{PUBCHEM_BASE}/compound/xref/RN/{urllib.parse.quote(cas)}/property/IsomericSMILES,CanonicalSMILES/JSON"
    j, _ = _get_json(url)
    return _extract_smiles_from_property(j) if j else None

def pc_props_from_name(name: str) -> Optional[str]:
    name = (name or "").strip()
    if not name:
        return None
    url = f"{PUBCHEM_BASE}/compound/name/{urllib.parse.quote(name)}/property/IsomericSMILES,CanonicalSMILES/JSON"
    j, _ = _get_json(url)
    return _extract_smiles_from_property(j) if j else None

def pc_cid_from_identifier(identifier: str) -> Optional[int]:
    ident = (identifier or "").strip()
    if not ident:
        return None
    url = f"{PUBCHEM_BASE}/compound/name/{urllib.parse.quote(ident)}/cids/JSON"
    j, _ = _get_json(url)
    try:
        return j["IdentifierList"]["CID"][0] if j else None
    except Exception:
        return None

def pc_smiles_from_cid_or_pugview(cid: int) -> Optional[str]:
    # try property first
    url = f"{PUBCHEM_BASE}/compound/cid/{cid}/property/IsomericSMILES,CanonicalSMILES/JSON"
    j, _ = _get_json(url)
    smi = _extract_smiles_from_property(j) if j else None
    if smi:
        return smi
    # fallback: PUG-View
    url = f"{PUGVIEW_BASE}/{cid}/JSON/"
    j, _ = _get_json(url)
    return _extract_smiles_from_pugview(j) if j else None

def lookup_pubchem_isomeric_smiles(cas_number: str, principal_name: str) -> Optional[str]:
    # 1) CAS 
    smi = pc_props_from_RN(cas_number)
    if smi:
        return smi
    # 2) Name 
    smi = pc_props_from_name(principal_name)
    if smi:
        return smi
    # 3) name→CID, then CAS→CID
    for ident in (principal_name, cas_number):
        cid = pc_cid_from_identifier(ident)
        if cid:
            smi = pc_smiles_from_cid_or_pugview(cid)
            if smi:
                return smi
    return None

# ───── Main ─────
def main():
    df = pd.read_csv(IN_CSV)

    req = ["CAS number", "Principal name"]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise SystemExit(f"Missing required columns: {missing}")

    smiles_cache = {}
    smiles_col = []

    for i, row in df.iterrows():
        cas  = row.get("CAS number", "")
        name = row.get("Principal name", "")
        key = (str(cas).strip(), str(name).strip())

        if key in smiles_cache:
            smi = smiles_cache[key]
        else:
            smi = lookup_pubchem_isomeric_smiles(cas, name)
            smiles_cache[key] = smi
            time.sleep(SLEEP_BETWEEN_CALLS)

        smiles_col.append(smi if smi else "")

        # light progress
        if (i + 1) % 50 == 0 or i == len(df) - 1:
            print(f"[{i+1}/{len(df)}] resolved so far", flush=True)

    # Insert *only* PUBCHEM_ISOMERIC_SMILES and write out
    insert_at = df.columns.get_loc("Principal name") + 1
    df.insert(insert_at, "PUBCHEM_ISOMERIC_SMILES", smiles_col)

    df.to_csv(OUT_CSV, index=False)
    print(f"\nDone. Wrote: {OUT_CSV}")

if __name__ == "__main__":
    main()


[50/3119] resolved so far
[100/3119] resolved so far
[150/3119] resolved so far
[200/3119] resolved so far
[250/3119] resolved so far
[300/3119] resolved so far
[350/3119] resolved so far
[400/3119] resolved so far
[450/3119] resolved so far
[500/3119] resolved so far
[550/3119] resolved so far
[600/3119] resolved so far
[650/3119] resolved so far
[700/3119] resolved so far
[750/3119] resolved so far
[800/3119] resolved so far
[850/3119] resolved so far
[900/3119] resolved so far
[950/3119] resolved so far
[1000/3119] resolved so far
[1050/3119] resolved so far
[1100/3119] resolved so far
[1150/3119] resolved so far
[1200/3119] resolved so far
[1250/3119] resolved so far
[1300/3119] resolved so far
[1350/3119] resolved so far
[1400/3119] resolved so far
[1450/3119] resolved so far
[1500/3119] resolved so far
[1550/3119] resolved so far
[1600/3119] resolved so far
[1650/3119] resolved so far
[1700/3119] resolved so far
[1750/3119] resolved so far
[1800/3119] resolved so far
[1850/3119] 